In [ ]:
# %pip install icecream
# %pip install scikit-image

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Imports:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import xarray as xr
import cf_xarray as cfxr
from pathlib import Path
import cftime
import pandas as pd
import matplotlib
from pprint import pprint
import importlib

import sys

from urclimask.urban_areas import (
    UrbanVicinity
)
from urclimask.utils import (
    kelvin2degC,
    traverseDir,
    fix_360_longitudes,
    RCM_DICT,
    load_ucdb_city
)

from urclimask.UHI_analysis import (
    UrbanIsland
)
from urclimask.GHCNd_stations import (
    load_ghcnd_stations,
    get_ghcnd_df,
    get_valid_timeseries,
    available_vars,
    inside_city
)

import glob

In [7]:
##Variables:

# path1 = '/lustre/gmeteo/ASNA/DATA/I4C/CMIP6/DD/ALPX-3f/BCCR-UCAN/ERA5/evaluation/r1i1p1f1/WRF451R-CI4C/v1-r1/6hr/ta1000/v20240710'
path = ''
file = 'ta1000_ALPX-3f_ERA5_evaluation_r1i1p1f1_BCCR-UCAN_WRF451R-CI4C_v1-r1_6hr_202301010000-202312311800.nc'

city = "Paris"
lon_city = 2.35
lat_city = 48.85
# variable, domain, reanalisis, simulation_tipe, run_type1, runing_centre, model, run_type2, time_interval, time_period = file.split("_")
variable, domain, driving_model, scenario, member, institution, model, version, time_interval, time_period = file.split("_")

# domain_resolution = 3 #??
urban_var = 'sftimf'

#Recomended parameters
urban_th = 60
urban_sur_th = 15
orog_diff = 100
sftlf_th = 70
ratio_r2u = 1 #o 2
min_city_size = 80 #Adjust depending on the city size, for Paris 80 is a good value, for smaller cities it should be smaller and for bigger cities 
                    #it should be bigger. It is the minimum number of grid points that the city should have to be considered in the analysis.
lon_lim = 0.7
lat_lim = 0.7

# output_dir = Path("results") / f"{city}-{domain}_{model}_{variable}"
# output_dir.mkdir(parents=True, exist_ok=True)

output_dir = 'Results'

domain_resolution = 3
base_filename = f'{city}-{domain}_ECMWF-ERA5_{scenario}_r1i1p1f1_{model}'




In [14]:
## Define urban areas and vicinity:

# root_nextcloud = '/.../CORDEX-CORE-WG/'#It defines 2 roots
# root_esgf = '/.../cordex/output/'
root_fields = ':/lustre/gmeteo/DATA/oceano/gmeteo/'#Root where the fields are stored: sfturf, orog, sftlf.

#Search for the files in the defined roots
file_sfturf = glob.glob(#It searches for all the .nc files that match this pattern
        f"{root_fields}{model}/{urban_var}/{urban_var}_{domain}*.nc" 
        # f"{root_nextcloud}{model}/{urban_var}/{urban_var}_{domain}*.nc" #"root/with/patern/*.nc"
)

file_orog = glob.glob(
    f"{root_fields}{domain}/.../ECMWF-ERAINT/evaluation/*/.../*/fx/orog/*/orog_*.nc" 
    # f"{root_esgf}{domain}/{RCM_DICT[domain][model].split('_')[0]}/ECMWF-ERAINT/evaluation/*/{RCM_DICT[domain][model].split('_')[1]}/*/fx/orog/*/orog_*.nc" 
)
file_sftlf = glob.glob(
    f"{root_fields}{domain}/.../ECMWF-ERAINT/evaluation/*/.../*/fx/sftlf/*/sftlf_*.nc" 
)





In [18]:
#No hay coincidencias con la ruta
# file_sfturf = []
# file_orog = []
# file_sftlf = []

[]

In [15]:
ds_sfturf = xr.open_dataset(file_sfturf[0])
ds_orog = xr.open_dataset(file_orog[0])
ds_sftlf = xr.open_dataset(file_sftlf[0])

ds_sfturf = fix_360_longitudes(ds_sfturf)#función del repositorio urclimask, definida en urclimask/utils.py. 
                                        #Sirve para convertir longitudes en formato 0–360° al formato -180–180°
ds_orog = fix_360_longitudes(ds_orog)
ds_sftlf = fix_360_longitudes(ds_sftlf)

IndexError: list index out of range

In [19]:
URBAN = UrbanVicinity(
    urban_sur_th = urban_sur_th,
    orog_diff = orog_diff,
    sftlf_th = sftlf_th,
    ratio_r2u = ratio_r2u,
    min_city_size = min_city_size,
    lon_city = lon_city,
    lat_city = lat_city,
    lon_lim = lon_lim,
    lat_lim = lat_lim,
    model = model,
    domain = domain,
    urban_th = urban_th,
    urban_var = urban_var
)

In [ ]:
# Crop area around de city in the fields datasets
ds_sfturf = URBAN.crop_area_city(ds = ds_sfturf, res = domain_resolution)
ds_orog = URBAN.crop_area_city(ds = ds_orog, res = domain_resolution)
ds_sftlf = URBAN.crop_area_city(ds = ds_sftlf, res = domain_resolution)

In [ ]:
# Define masks using the parameters above
sfturf_mask, sfturf_sur_mask, orog_mask, sftlf_mask = URBAN.define_masks(
    ds_sfturf = ds_sfturf, #
    ds_orog = ds_orog, 
    ds_sftlf = ds_sftlf,
)

# sfturf_mask: Urban mask, identifies urban grid cells based on the urban variable (sftimf) and the defined urban threshold (urban_th).
# sfturf_sur_mask: Urban surroundings mask, identifies grid cells in the transition zone between urban and rural areas based on the urban surface threshold (urban_sur_th).
# orog_mask: Orography mask, identifies grid cells where the elevation difference is less than the defined threshold (orog_diff). 


In [ ]:
## Final rural-urban mask:
# Starting from the city, it progressively expands the surrounding area until it finds enough comparable rural cells, 
# excluding sea, areas with non-comparable orography, and peri-urban zones.
urmask = URBAN.select_urban_vicinity(
    sfturf_mask = sfturf_mask, 
    orog_mask = orog_mask,
    sftlf_mask = sftlf_mask,
    sfturf_sur_mask = sfturf_sur_mask
)
#ratio_r2u-> Criteria for enough riral cells, predefined (self)


In [ ]:
#In represents the original variable filtered by the masks
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                 composite = False)
# fig.savefig(f"{output_dir}/urmask_{base_filename}_fx.pdf", bbox_inches='tight')

In [ ]:
# Mask's final result: Urban centre and rural surroundings
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                 composite = True)
fig.savefig(f"{output_dir}/urmask_{base_filename}_fx.pdf", bbox_inches='tight')
# fig.savefig(f"{output_dir}/urmask_{base_filename}_fx.pdf", bbox_inches='tight')#Save figure
# urmask.to_netcdf(f"{output_dir}/urmask_{base_filename}_fx.nc")#Save to netcdf

In [ ]:
#calculate urban heat island (UHI)
files_pattern = f"{root_esgf}{domain}/{RCM_DICT[domain][model].split('_')[0]}/*/{scenario}/*/{RCM_DICT[domain][model].split('_')[1]}/*/day/{variable}/*/{variable}_*.nc"
files = glob.glob(files_pattern)
ds_RCM = xr.open_mfdataset(sorted(files), combine='nested', concat_dim='time')#It opens multiple nc files as an unic dataset

ds_RCM = kelvin2degC(ds_RCM, variable)#K->ºC
ds_RCM = fix_360_longitudes(ds_RCM)#0–360 -> -180–180
ds_RCM = URBAN.crop_area_city(ds = ds_RCM, res = domain_resolution)#crop to the city area

# It loads the city polygon from the UCDB database
root_nextcloud = '/.../CORDEX-CORE-WG/'
ucdb_city = load_ucdb_city(root_nextcloud, city)

Then it compares results with observations

In [ ]:
file_path = '.../PARIS_surface_weather_data/combined_temp_data.csv'
combined_data = pd.read_csv(file_path)

combined_data['date'] = pd.to_datetime(combined_data['DATE'])#Creates a new column called date from the original DATE column, 
                                                                # converting it to pandas datetime format.
combined_data.drop(columns=['DATE'], inplace=True)#It eliminates the original DATE column
combined_data = combined_data.set_index('date')#It converts the date column into the index of the dataframe
combined_data['code'] = combined_data['code'].astype(int)#Ir converts the code column to integer type

¿donde obtener cada cosa?

GHS-UCDB / Copernicus Human Settlement
- polígonos y atributos urbanos (para localizar/representar la ciudad)
- lo tengo que sacar de la página 'https://human-settlement.emergency.copernicus.eu/ghs_ucdb_2024.php'?

sfturf, orog, sftlf
- variables estáticas del modelo
- He encontrado que están en /lustre/gmeteo/DATA/oceano/gmeteo, pero no tengo acceso
- Tampoco tengo acceso al otro servidor (HUB)


observaciones
- datos de temperatura observada (comparar modelo vs observaciones)